# 回归模型对比

这份 notebook 不只是跑线性回归，而是在比较不同回归模型如何处理同一组房价数据。

## 回归任务关注什么
- 预测值和真实值差多远。
- 模型是否容易过拟合。
- 特征系数是否稳定、是否容易解释。


In [109]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, mean_squared_error, roc_auc_score


import numpy as np
import pandas as pd
import joblib

import os

## 为什么房价数据适合做回归入门

目标值是连续变量，特征也比较标准，非常适合用来比较不同回归器的行为差异。


In [110]:
# Load the California housing dataset
fe_cal = fetch_california_housing(data_home='data')

print(fe_cal.DESCR)

print("特征值\n", fe_cal.data[:5])
print("目标值\n", fe_cal.target[:5])

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

## 标准化对线性模型尤其重要

像 SGD、岭回归、Lasso 这类模型，对特征尺度比较敏感。先标准化，才能更公平地比较不同特征的作用，也让正则化项的惩罚更均衡。


In [111]:
x_train, x_test, y_train, y_test = train_test_split(
    fe_cal.data, fe_cal.target, test_size=0.25, random_state=42)

# 标准化
std = StandardScaler()
x_train = std.fit_transform(x_train)
x_test = std.transform(x_test)

## 线性回归的基本假设

线性回归默认目标和特征之间大致满足线性关系：
\[
y = \beta_0 + \beta_1x_1 + \cdots + \beta_nx_n + \epsilon
\]

它的优点是简单、可解释，但如果关系明显非线性，效果就会受限。


In [112]:
# 线性回归模型
# 原理：线性回归模型试图找到一个线性函数，使得输入特征与目标变量之间的关系尽可能接近。
# 它通过最小化预测值与实际值之间的误差来拟合数据。
# 公式：y = β0 + β1*x1 + β2*x2 + ... + βn*xn + ε，其中β0是截距，β1, β2, ..., βn是特征的系数，ε是误差项。
# 正则方程：β = (X^T * X)^(-1) * X^T * y，其中X是特征矩阵，y是目标变量向量，β是回归系数向量。

model = LinearRegression()
model.fit(x_train, y_train)

print("回归系数\n", model.coef_)
print("截距\n", model.intercept_)

y_pred = model.predict(x_test)

if os.path.exists('model/linear_regression_model.pkl'):
    os.unlink('model/linear_regression_model.pkl')

joblib.dump(model, 'model/linear_regression_model.pkl')

print("均方误差\n", mean_squared_error(y_test, y_pred))

回归系数
 [ 0.85210815  0.12065533 -0.30210555  0.34860575 -0.00164465 -0.04116356
 -0.89314697 -0.86784046]
截距
 2.0703489205424743
均方误差
 0.5411287478470689


## SGD、Ridge、Lasso 分别在解决什么

- SGDRegressor：用迭代优化替代一次性求解，适合更大规模数据。
- Ridge：在损失里加入 L2 正则，抑制系数过大，缓解共线性问题。
- Lasso：加入 L1 正则，除了防过拟合，还可能把一部分系数直接压成 0，起到特征选择作用。


In [113]:
# 标准化 target
# std_target = StandardScaler()
# y_train = std_target.fit_transform(y_train.reshape(-1, 1)).flatten()

# model.fit(x_train, y_train)
# y_pred = model.predict(x_test)

# # 将预测值反标准化
# y_pred = std_target.inverse_transform(y_pred.reshape(-1, 1)).flatten()


# print("回归系数\n", model.coef_)
# print("截距\n", model.intercept_)
# print("均方误差\n", mean_squared_error(y_test, y_pred))

In [114]:
# 梯度下降
# 原理：梯度下降是一种优化算法，用于最小化损失函数。它通过计算损失函数相对于模型参数的梯度，并沿着梯度的反方向更新参数来逐步逼近最优解。
# 公式：θ = θ - α * ∇J(θ)，其中θ是模型参数，α是学习率，∇J(θ)是损失函数J(θ)关于参数θ的梯度。
# 损失函数：常用的损失函数是均方误差（MSE），定义为J(θ) = (1/n) * Σ(y_i - h_θ(x_i))^2，其中n是样本数量，y_i是实际值，h_θ(x_i)是预测值。

# 与线性回归模型相比，SGDRegressor使用随机梯度下降算法来优化模型参数。
# 它在每次迭代中使用一个样本或一个小批量样本来计算梯度，从而加快了训练过程，特别适用于大规模数据集。

# 参数说明：
# max_iter：最大迭代次数，控制算法的收敛程度。
# tol：容忍度，控制算法的停止条件，当损失函数的变化小于tol时停止迭代。
# eta0：学习率，控制参数更新的步长。
# learning_rate：学习率的更新策略
# 常见的选项包括'constant'（固定学习率）、'optimal'（根据数据自动调整学习率）和'invscaling'（随着迭代次数增加逐渐减小学习率）。

sgd = SGDRegressor(eta0=0.01, max_iter=1000, tol=1e-3, random_state=42)
sgd.fit(x_train, y_train)

y_pred = sgd.predict(x_test)

print("回归系数\n", sgd.coef_)
print("截距\n", sgd.intercept_)
print("均方误差\n", mean_squared_error(y_test, y_pred))

回归系数
 [ 0.83264178  0.12498833 -0.25255367  0.4110813   0.00160827 -0.05168275
 -0.88794249 -0.87366852]
截距
 [2.05443863]
均方误差
 0.5854346671603641


In [115]:
# 正则化指的是在损失函数中添加一个惩罚项，以防止模型过拟合。常见的正则化方法有L1正则化和L2正则化。
# L1 L2 正则化
# L1正则化（Lasso回归）通过在损失函数中添加特征系数的绝对值来惩罚模型复杂度。
# 这会导致一些特征的系数变为零，从而实现特征选择。
# J(θ) = MSE(θ) + α * ||θ||_1，其中MSE是均方误差，α是正则化强度，||θ||_1是参数θ的L1范数（即参数的绝对值之和）。
# L2正则化（Ridge回归）通过在损失函数中添加特征系数的平方来惩罚模型复杂度。
# 这会使得特征系数趋向于零，但不会完全变为零。
# J(θ) = MSE(θ) + α * ||θ||_2^2，其中MSE是均方误差，α是正则化强度，||θ||_2^2是参数θ的L2范数的平方（即参数的平方和）。


# 两者的区别在于L1正则化会产生稀疏模型（即一些特征的系数为零），适用于特征选择；
# 而L2正则化会产生非稀疏模型（即所有特征的系数都趋向于零但不完全为零），适用于处理多重共线性问题。

In [116]:
# Lasso回归

# 参数说明：
# alpha：正则化强度，控制模型复杂度。较大的alpha会增加正则化效果，使得更多的特征系数趋向于零。
ls = Lasso(alpha=0.001)

ls.fit(x_train, y_train)
y_pred = ls.predict(x_test)

print("回归系数\n", ls.coef_)
print("截距\n", ls.intercept_)
print("均方误差\n", mean_squared_error(y_test, y_pred))

回归系数
 [ 8.46798957e-01  1.21459609e-01 -2.88593738e-01  3.35037689e-01
 -3.95220068e-04 -4.02168154e-02 -8.82194884e-01 -8.56223079e-01]
截距
 2.0703489205424765
均方误差
 0.5399841928119937


## 评价指标不只看一个

回归常见指标包括：
- MSE / RMSE：误差大小。
- MAE：平均绝对偏差，更抗异常值。
- R²：解释了多少目标方差。

不同指标强调的侧重点不同，所以对比模型时最好结合起来看。


In [117]:
# Ridge回归

# 参数说明：
# alpha：正则化强度，控制模型复杂度。
# 较大的alpha会增加正则化效果，导致更多的特征系数趋向于零。
rd = Ridge(alpha=0.02)
rd.fit(x_train, y_train)

print("回归系数\n", rd.coef_)
print("截距\n", rd.intercept_)
print("均方误差\n", mean_squared_error(y_test, rd.predict(x_test)))

回归系数
 [ 0.85210684  0.12065698 -0.30210097  0.3486001  -0.00164411 -0.04116364
 -0.89313096 -0.86782423]
截距
 2.0703489205424743
均方误差
 0.5411281184476519


---

In [118]:
column = ['Sample code number', 'Clump Thickness', 'Uniformity of Cell Size', 'Uniformity of Cell Shape',
          'Marginal Adhesion', 'Single Epithelial Cell Size', 'Bare Nuclei', 'Bland Chromatin',
          'Normal Nucleoli', 'Mitoses', 'Class']
data = pd.read_csv('data/breast-cancer-wisconsin.csv', names=column)

In [119]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Sample code number           699 non-null    int64
 1   Clump Thickness              699 non-null    int64
 2   Uniformity of Cell Size      699 non-null    int64
 3   Uniformity of Cell Shape     699 non-null    int64
 4   Marginal Adhesion            699 non-null    int64
 5   Single Epithelial Cell Size  699 non-null    int64
 6   Bare Nuclei                  699 non-null    str  
 7   Bland Chromatin              699 non-null    int64
 8   Normal Nucleoli              699 non-null    int64
 9   Mitoses                      699 non-null    int64
 10  Class                        699 non-null    int64
dtypes: int64(10), str(1)
memory usage: 60.2 KB


In [120]:
data.describe(include='all')

,Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
count,6.990000e+02,699.000000,699.000000,699.000000,699.000000,699.000000,699,699.000000,699.000000,699.000000,699.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,402,NaN,NaN,NaN,NaN
mean,1.071704e+06,4.417740,3.134478,3.207439,2.806867,3.216023,NaN,3.437768,2.866953,1.589413,2.689557
std,6.170957e+05,2.815741,3.051459,2.971913,2.855379,2.214300,NaN,2.438364,3.053634,1.715078,0.951273
min,6.163400e+04,1.000000,1.000000,1.000000,1.000000,1.000000,NaN,1.000000,1.000000,1.000000,2.000000
25%,8.706885e+05,2.000000,1.000000,1.000000,1.000000,2.000000,NaN,2.000000,1.000000,1.000000,2.000000
50%,1.171710e+06,4.000000,1.000000,1.000000,1.000000,2.000000,NaN,3.000000,1.000000,1.000000,2.000000
75%,1.238298e+06,6.000000,5.000000,5.000000,4.000000,4.000000,NaN,5.000000,4.000000,1.000000,4.000000


In [121]:
data.replace('?', np.nan, inplace=True)
data.dropna(inplace=True)
data.info()

<class 'pandas.DataFrame'>
Index: 683 entries, 0 to 698
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Sample code number           683 non-null    int64
 1   Clump Thickness              683 non-null    int64
 2   Uniformity of Cell Size      683 non-null    int64
 3   Uniformity of Cell Shape     683 non-null    int64
 4   Marginal Adhesion            683 non-null    int64
 5   Single Epithelial Cell Size  683 non-null    int64
 6   Bare Nuclei                  683 non-null    str  
 7   Bland Chromatin              683 non-null    int64
 8   Normal Nucleoli              683 non-null    int64
 9   Mitoses                      683 non-null    int64
 10  Class                        683 non-null    int64
dtypes: int64(10), str(1)
memory usage: 64.0 KB


In [122]:
data[column[6]] = data[column[6]].astype('int64')
data.info()

<class 'pandas.DataFrame'>
Index: 683 entries, 0 to 698
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Sample code number           683 non-null    int64
 1   Clump Thickness              683 non-null    int64
 2   Uniformity of Cell Size      683 non-null    int64
 3   Uniformity of Cell Shape     683 non-null    int64
 4   Marginal Adhesion            683 non-null    int64
 5   Single Epithelial Cell Size  683 non-null    int64
 6   Bare Nuclei                  683 non-null    int64
 7   Bland Chromatin              683 non-null    int64
 8   Normal Nucleoli              683 non-null    int64
 9   Mitoses                      683 non-null    int64
 10  Class                        683 non-null    int64
dtypes: int64(11)
memory usage: 64.0 KB


In [123]:
x_train, x_test, y_train, y_test = train_test_split(
    data[column[1:10]], data[column[10]], test_size=0.25, random_state=42)

std = StandardScaler()
x_train = std.fit_transform(x_train)
x_test = std.transform(x_test)

x_train

array([[-0.1467477 , -0.69257321, -0.73219813, ..., -0.99166968,
        -0.5989596 , -0.34508231],
       [-0.50624446, -0.69257321, -0.40450106, ..., -0.56820439,
        -0.5989596 , -0.34508231],
       [ 2.01023286,  2.24337652,  2.21707545, ...,  0.27872617,
        -0.5989596 , -0.34508231],
       ...,
       [ 2.01023286,  0.61229334,  1.23398426, ..., -0.14473911,
         0.07801019,  3.934416  ],
       [-0.1467477 , -0.69257321, -0.73219813, ..., -0.56820439,
        -0.5989596 , -0.34508231],
       [ 2.01023286,  2.24337652,  2.21707545, ...,  1.97258731,
         1.77043467,  3.934416  ]], shape=(512, 9))

In [124]:
# 逻辑回归
# 逻辑回归是一种用于二分类问题的线性模型。
# 它通过使用sigmoid函数将线性组合的输入特征映射到0和1之间的概率值，从而进行分类。
# 公式：P(y=1|x) = σ(β0 + β1*x1 + β2*x2 + ... + βn*xn)
# 其中σ(z)是sigmoid函数，β0是截距，β1, β2, ..., βn是特征的系数，x1, x2, ..., xn是输入特征。
# sigmoid函数：σ(z) = 1 / (1 + exp(-z))，其中z是线性组合的输入特征。
# 逻辑回归的损失函数是对数损失函数（log loss），它衡量了模型预测概率与实际标签之间的差异。
# cost = - (y * log(p) + (1 - y) * log(1 - p))，其中y是实际标签，p是模型预测的概率。
# 逻辑回归的目标是最大化对数似然函数，或者等价地最小化对数损失函数，从而找到最佳的模型参数。
# 逻辑回归的决策边界是一个线性超平面，它将特征空间划分为两个部分，分别对应于不同的类别。
# 模型通过学习参数来调整这个决策边界，以便更好地分类数据点。

# 参数说明：
# C：正则化强度的倒数，控制模型复杂度。较小的C会增加正则化效果，使得模型更简单。
# solver：优化算法，控制模型参数的求解方式。
# 常见的选项包括'lbfgs'（拟牛顿法）、'liblinear'（坐标下降法）和'sag'（随机平均梯度下降法）。

lg = LogisticRegression(C=0.5, solver='lbfgs')

lg.fit(x_train, y_train)
print("回归系数\n", lg.coef_)
print("截距\n", lg.intercept_)
y_pred = lg.predict(x_test)

print('准确率\n', lg.score(x_test, y_test))
print('分类报告\n', classification_report(y_test, y_pred,
      labels=[2, 4], target_names=['良性', '恶性']))
print('AUC\n', roc_auc_score(y_test, y_pred))

回归系数
 [[1.05630745 0.33333847 0.73928032 0.5087008  0.15545045 1.25353493
  0.9515665  0.43845323 0.52168951]]
截距
 [-1.49916812]
准确率
 0.9532163742690059
分类报告
               precision    recall  f1-score   support

          良性       0.94      0.99      0.96       103
          恶性       0.98      0.90      0.94        68

    accuracy                           0.95       171
   macro avg       0.96      0.94      0.95       171
weighted avg       0.95      0.95      0.95       171

AUC
 0.943675042832667


## 学习检查清单

完成本 notebook 后，可以尝试回答：
- 这个任务属于监督学习、无监督学习，还是特征处理流程？
- 当前算法依赖哪些关键假设，什么数据场景下可能失效？
- 代码中哪些步骤是在防止数据泄漏或过拟合？
- 如果指标不理想，应该优先调整特征、模型参数还是评估方式？
